# Rootograms for count regressions

## Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import poisson, nbinom
import matplotlib.pyplot as plt

## Data

In [ ]:
from pathlib import Path
csv_path = Path("data/polish-jvs.csv")
if not csv_path.exists():
    csv_path = Path("../../data/polish-jvs.csv")

df = pd.read_csv(csv_path,
                 dtype={"id": np.int64, "woj": str, "public": str,
                        "size": str, "nace_division": str, "nace": str})
df["size"] = pd.Categorical(df["size"], categories=["Large", "Medium", "Small"])

## Fit Poisson and Negative Binomial models

In [ ]:
pois = smf.glm("vacancies ~ C(size) + C(public) + C(nace)", data=df,
               family=sm.families.Poisson()).fit()

# Joint MLE for NB2 (matches MASS::glm.nb)
start = np.append(pois.params.values, 1.0)
nb2 = smf.negativebinomial("vacancies ~ C(size) + C(public) + C(nace)",
                           data=df).fit(start_params=start,
                                        method="bfgs", maxiter=500, disp=0)

## Python — manual implementation

In [ ]:
def rootogram(model, y, max_count=20, dist="poisson", ax=None,
              title=None):
    """Hanging rootogram for a fitted Poisson or NB2 GLM.

    model : statsmodels results object with .predict() returning fitted means
    y     : observed response (1d array-like of non-negative ints)
    dist  : "poisson" or "nbinom"
    """
    mu = np.asarray(model.predict())
    y  = np.asarray(y)
    ks = np.arange(max_count + 1)

    observed = np.array([(y == k).sum() for k in ks], dtype=float)

    if dist == "poisson":
        expected = np.array([poisson.pmf(k, mu).sum() for k in ks])
    elif dist == "nbinom":
        alpha = float(model.params["alpha"])
        n = 1.0 / alpha
        p = n / (n + mu)
        expected = np.array([nbinom.pmf(k, n, p).sum() for k in ks])
    else:
        raise ValueError("dist must be 'poisson' or 'nbinom'")

    obs_root = np.sqrt(observed)
    exp_root = np.sqrt(expected)

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))

    ax.bar(ks, obs_root, bottom=exp_root - obs_root, width=0.7,
           color="lightgray", edgecolor="black", label="observed")
    ax.plot(ks, exp_root, "ro-", linewidth=1.5, markersize=4,
            label=r"$\sqrt{\mathrm{expected}}$")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("count")
    ax.set_ylabel(r"$\sqrt{\mathrm{frequency}}$")
    ax.set_xticks(ks[::2])
    if title:
        ax.set_title(title)
    ax.legend(loc="upper right")
    return ax

In [ ]:
#| fig-width: 8
#| fig-height: 4
fig, ax = plt.subplots(figsize=(8, 4))
rootogram(pois, df["vacancies"], dist="poisson", ax=ax, title="Poisson")
plt.show()

In [ ]:
#| fig-width: 8
#| fig-height: 4
fig, ax = plt.subplots(figsize=(8, 4))
rootogram(nb2, df["vacancies"], dist="nbinom", ax=ax,
          title="Negative binomial (NB2)")
plt.show()

## Numerical comparison

In [ ]:
ks = np.arange(11)
obs = np.array([(df["vacancies"] == k).sum() for k in ks], dtype=float)

mu_p  = pois.predict()
exp_p = np.array([poisson.pmf(k, mu_p).sum() for k in ks])

alpha = float(nb2.params["alpha"])
n_par = 1.0 / alpha
mu_n  = nb2.predict()
p_par = n_par / (n_par + mu_n)
exp_n = np.array([nbinom.pmf(k, n_par, p_par).sum() for k in ks])

pd.DataFrame({"k": ks,
              "observed":  obs,
              "exp_pois":  exp_p,
              "exp_nb2":   exp_n,
              "diff_pois": obs - exp_p,
              "diff_nb2":  obs - exp_n}).round(1)